[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/building-agents/notebooks/m2_tools_memory.ipynb) [![Course](https://img.shields.io/badge/Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/building-agents#module-2)

# Module 2 — Tools & Memory

**Level:** Beginner | **Time:** ~45 min | [Full reading →](https://rajeevraibhatia.com/curriculum/building-agents#module-2)

### Key concepts
- Tool descriptions are the model's only docs — make them verbose
- Conversation history = the `messages` list. Keep it between turns.
- Return errors as strings, not exceptions

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Option A: OpenAI API (recommended for Colab)
!pip install openai --quiet

import os
from openai import OpenAI

# Set your OpenAI API key — in Colab: Secrets (🔑) → add OPENAI_API_KEY
# Then enable notebook access, or paste directly (don't commit keys to git)
# os.environ["OPENAI_API_KEY"] = "sk-..."   # ← uncomment and paste if not using Secrets

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o"

# ── Option B: Ollama (local, no API key, no cost) ─────────────────────────────
# 1. Install Ollama: https://ollama.com/download
# 2. Run: ollama pull llama3.2
# 3. Uncomment the two lines below and comment out the OpenAI lines above:
#
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# MODEL = "llama3.2"   # or: mistral, phi4, gemma3, qwen2.5, etc.
#
# Everything in this notebook works with either client — MODEL is passed through.
print(f"Client ready. Using model: {MODEL}")

In [ ]:
import json
import math
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_notes: list[str] = []

TOOLS = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a math expression. Example: '847 * 0.15'",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string"}}, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "save_note",
        "description": "Save a text note for later retrieval.",
        "parameters": {"type": "object",
            "properties": {"text": {"type": "string"}}, "required": ["text"]}
    }},
    {"type": "function", "function": {
        "name": "get_notes",
        "description": "Retrieve all saved notes.",
        "parameters": {"type": "object", "properties": {}}
    }},
]

def run_tool(name: str, args: dict) -> str:
    if name == "calculator":
        try:
            return str(eval(args["expression"], {"__builtins__": {}}, vars(math)))
        except Exception as e:
            return f"Error: {e}"
    if name == "save_note":
        _notes.append(args["text"])
        return f"Note saved (#{len(_notes)})."
    if name == "get_notes":
        return "\n".join(f"{i+1}. {n}" for i, n in enumerate(_notes)) if _notes else "No notes yet."
    return f"Unknown tool: {name}"

def run_agent(user_input: str, messages: list) -> str:
    messages.append({"role": "user", "content": user_input})
    while True:
        r = client.chat.completions.create(model="gpt-4o", messages=messages, tools=TOOLS)
        msg = r.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            result = run_tool(tc.function.name, json.loads(tc.function.arguments))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

history = [{"role": "system", "content": "You are a helpful assistant."}]
print(run_agent("What is 15% of 847? Save the result as a note.", history))
print(run_agent("What notes have I saved so far?", history))

## Exercise

Add a `delete_note(index)` tool. Handle out-of-range gracefully.

In [ ]:
# Exercise: add delete_note(index) tool
# index is 1-based (first note = 1)
# Return an error string if index is out of range (don't raise!)
# Test: save 3 notes, delete the second, call get_notes

# Your code here: